# NB02b — OU Pairs Portfolio (Appendix / Exploration)

**Role.** This is a companion *appendix* to NB02. It reads the frozen
statistical-baseline artifacts exported by NB02 and explores how the
per-pair OU PnL aggregates into a multi-pair portfolio. It is intentionally
kept **out of the NB02 -> NB03 contract**: NB03 consumes the per-pair
artifacts, not these portfolio aggregates.

**Why separate.** Portfolio construction here is on the *unconditioned* OU
baseline. The deployable, capital-scaled portfolio — built on the
**AI-conditioned** book — is the job of NB04. Treat the results below as a
benchmark for the raw baseline, not a deployable allocation.

**What this notebook fixes.** The naive equal-weight portfolio averages raw
spread-dollar PnL across pairs whose dollar scales differ by orders of
magnitude (β ranges from ~0.03 to ~70, so the spread `S = y − (α+βx)` lives
on wildly different scales). Naive EW is therefore dominated by a few
high-β pairs and is **not** equal-risk. This appendix makes the scale
problem explicit and rebuilds the portfolio on a risk-normalized basis.

In [ ]:
# ==============================================================================
# 0. Environment + project-root anchor + load NB02 artifacts
# ==============================================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

plt.style.use("seaborn-v0_8")
sns.set_theme(context="notebook", style="whitegrid")

def find_project_root(markers=("pyproject.toml", ".git")):
    here = Path.cwd().resolve()
    for cand in (here, *here.parents):
        if any((cand / m).exists() for m in markers):
            return cand
    return here

ROOT      = find_project_root()
ART       = ROOT / "artifacts" / "nb2_outputs"
ANNUAL_D  = 252

# --- Load artifacts (read-only handoff from NB02) ---
pair_ts = pd.read_parquet(ART / "pair_timeseries.parquet")
trades  = pd.read_csv(ART / "trades_table.csv", parse_dates=["entry_date", "exit_date"])
meta    = pd.read_csv(ART / "pairs_metadata.csv")

# Rebuild the wide per-pair PnL matrix (date × pair). NB02 exports PnL on the
# full OOS index with gaps already zero-filled, so this pivot is clean.
pnl_wide = (pair_ts.pivot(index="date", columns="pair", values="pnl")
                   .sort_index()
                   .fillna(0.0))

N = pnl_wide.shape[1]
print(f"[OK] Loaded artifacts from: {ART}")
print(f"[OK] pnl_wide: {pnl_wide.shape}  ({N} pairs × {len(pnl_wide)} OOS days)")
print(f"[OK] trades:   {trades.shape};  pairs in metadata: {len(meta)}")

## 1. The scale problem

Each pair's PnL is in *spread-dollar units per unit of position*, and the
spread scale is set by the hedge ratio β and the price levels of the legs.
The table below shows annualized per-pair PnL and volatility side-by-side
with β. If the dollar scales span orders of magnitude, a plain average over
pairs is dominated by the largest-scale names — so "equal weight" is really
"β-weight".

In [ ]:
# ==============================================================================
# 1. Per-pair scale table + naive-EW domination check
# ==============================================================================
ann_pnl_i = pnl_wide.mean() * ANNUAL_D
ann_vol_i = pnl_wide.std(ddof=0) * np.sqrt(ANNUAL_D)

scale_tbl = (pd.DataFrame({
                "beta":   meta.set_index("pair")["beta"],
                "AnnPnL": ann_pnl_i,
                "AnnVol": ann_vol_i,
             })
             .assign(AnnVol_share=lambda d: d["AnnVol"].abs() / d["AnnVol"].abs().sum())
             .sort_values("AnnVol", ascending=False, key=lambda s: s.abs()))

print("[INFO] Per-pair scale (sorted by |AnnVol|):")
print(scale_tbl.round(3).to_string())

# How concentrated is a NAIVE equal-weight average of raw PnL?
pnl_eq_naive = pnl_wide.mean(axis=1)
ann_share = (ann_pnl_i / ann_pnl_i.sum()).sort_values(ascending=False)
top3 = ann_share.head(3)
print(f"\n[INFO] Naive EW is dominated by a few high-scale pairs:")
print(f"       Top-3 pairs' share of summed AnnPnL: {top3.sum():.1%}")
for p, sh in top3.items():
    print(f"         {p:<14s}: {sh:6.1%}")
print(f"       Spread of |AnnVol| (max/min): "
      f"{ann_vol_i.abs().max() / ann_vol_i.abs().replace(0, np.nan).min():,.0f}x")

## 2. Risk-normalized portfolio construction

To aggregate meaningfully, scale each pair so it contributes comparable
**risk**, then weight. We show three risk-aware constructions against the
naive baseline:

- **Naive EW** — `mean` of raw spread-dollar PnL (the anti-pattern, shown for contrast).
- **Equal-risk (full-sample inv-vol)** — `wᵢ ∝ 1/σᵢ` using full-OOS σ. Textbook
  risk-parity-without-correlation benchmark; mildly in-sample-aware (uses the
  whole window to set weights).
- **Equal-risk (causal)** — each pair scaled by its *expanding, lagged* vol
  estimate (`σ` up to *t−1*), so weights use only past information — the
  deployable-style construction.
- **Vol-target overlay** — linear scaling of the causal equal-risk series to a
  10% annualized vol target (a leverage transform; preserves Sharpe).

In [ ]:
# ==============================================================================
# 2. Build portfolios
# ==============================================================================
def compute_portfolio_stats(pnl, label, annual_days=ANNUAL_D):
    pnl = pnl.fillna(0.0)
    mu  = pnl.mean() * annual_days
    vol = pnl.std(ddof=0) * np.sqrt(annual_days)
    sharpe = mu / vol if vol > 0 else np.nan
    equity = pnl.cumsum()
    max_dd = (equity - equity.cummax()).min()
    return {"Portfolio": label, "AnnPnL": mu, "AnnVol": vol,
            "Sharpe": sharpe, "MaxDD": max_dd}

# (a) Naive EW — raw spread-dollar PnL
pnl_eq_naive = pnl_wide.mean(axis=1)

# (b) Equal-risk, full-sample inverse-vol
pair_vol = pnl_wide.std(ddof=0).replace(0.0, np.nan)
w_invvol = (1.0 / pair_vol); w_invvol /= w_invvol.sum()
pnl_eqrisk_fs = (pnl_wide * w_invvol).sum(axis=1)

# (c) Equal-risk, CAUSAL (expanding, lagged vol -> no look-ahead).
# A naive 1/vol blows up early when a pair's vol estimate is ~0 (no trades yet),
# so floor each pair's vol at 10% of its full-sample vol. The floor is the ONLY
# place full-sample info enters; the per-bar scaling stays causal, and the floor
# only binds during the early, low-activity window.
vol_causal = pnl_wide.expanding(min_periods=63).std(ddof=0).shift(1)
vol_floor  = 0.10 * pnl_wide.std(ddof=0)                      # per-pair floor
vol_causal = vol_causal.clip(lower=vol_floor, axis=1)
unit_vol   = (pnl_wide / vol_causal).where(vol_causal.notna())
unit_vol   = unit_vol.replace([np.inf, -np.inf], np.nan)
pnl_eqrisk_causal = unit_vol.mean(axis=1).fillna(0.0)         # equal weight on unit-vol pairs

# (d) Vol-target overlay on the causal equal-risk series (10% ann vol)
TARGET_VOL = 0.10
realized = pnl_eqrisk_causal.std(ddof=0) * np.sqrt(ANNUAL_D)
scale = TARGET_VOL / realized if realized > 0 else 0.0
pnl_target = pnl_eqrisk_causal * scale

summary = pd.DataFrame([
    compute_portfolio_stats(pnl_eq_naive,      "Naive EW (raw spread-$)"),
    compute_portfolio_stats(pnl_eqrisk_fs,     "Equal-Risk (inv-vol, full-sample)"),
    compute_portfolio_stats(pnl_eqrisk_causal, "Equal-Risk (causal)"),
    compute_portfolio_stats(pnl_target,        f"Vol-Target {TARGET_VOL:.0%} (causal)"),
]).set_index("Portfolio")

print(f"[INFO] Causal equal-risk scaling factor to {TARGET_VOL:.0%}: {scale:.4f}")
print("\n[INFO] Portfolio summary (PnL in normalized units except Naive EW):")
print(summary.round(3).to_string())

In [ ]:
# ==============================================================================
# 2b. Equity curves — naive EW vs risk-normalized constructions
# ==============================================================================
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Naive EW on its own axis (different units, dominated by high-β pairs)
pnl_eq_naive.cumsum().plot(ax=ax1, color="crimson")
ax1.set_title("Naive EW (raw spread-$ units)\n— dominated by high-β pairs")
ax1.set_ylabel("Cumulative PnL (spread-$)")

# Risk-normalized constructions share comparable (unit-vol) scale
pnl_eqrisk_fs.cumsum().plot(ax=ax2, label="Equal-Risk (inv-vol, full-sample)")
pnl_eqrisk_causal.cumsum().plot(ax=ax2, label="Equal-Risk (causal)")
pnl_target.cumsum().plot(ax=ax2, label=f"Vol-Target {TARGET_VOL:.0%}", linestyle="--")
ax2.set_title("Risk-normalized portfolios (comparable units)")
ax2.set_ylabel("Cumulative normalized PnL")
ax2.legend()

plt.tight_layout(); plt.show()

## 3. Robustness & risk diagnostics

The diagnostics below operate on the **risk-normalized** pair PnL
(`unit_vol`), so cross-pair correlation, effective breadth, and contribution
are measured on comparable-risk units rather than raw spread dollars.

In [ ]:
# ==============================================================================
# 3A. Pair-level PnL correlation matrix (risk-normalized units)
# ==============================================================================
pnl_norm = unit_vol.fillna(0.0)   # unit-vol-scaled per-pair PnL
corr_pnl = pnl_norm.corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr_pnl, cmap="coolwarm", center=0, annot=False)
plt.title("Pair-Level PnL Correlation Matrix (risk-normalized)")
plt.tight_layout(); plt.show()

In [ ]:
# ==============================================================================
# 3A.1 Effective number of independent pairs
# ==============================================================================
Np = corr_pnl.shape[0]
mask = np.triu(np.ones((Np, Np), dtype=bool), k=1)
high_corr = (corr_pnl.values[mask] >= 0.5).sum()
total_offdiag = mask.sum()

eigvals = np.clip(np.linalg.eigvalsh(corr_pnl.values), 0, None)
n_eff = (eigvals.sum() ** 2) / (eigvals ** 2).sum()

print("[INFO] Pairwise correlation diagnostic (risk-normalized):")
print(f"       Off-diagonal pairs with rho >= 0.5:  {high_corr} / {total_offdiag}")
print(f"       Effective N (eigenvalue spread):     {n_eff:.2f} / {Np}")
print(f"       Diversification ratio:               {n_eff / Np:.2f}")

flat = corr_pnl.where(mask).stack().sort_values(ascending=False)
print("\n[INFO] Top 5 most correlated pair-pairs:")
for (a, b), rho in flat.head(5).items():
    print(f"       {a:<14s} <-> {b:<14s}: rho = {rho:.3f}")

In [ ]:
# ==============================================================================
# 3B. Rolling Sharpe — naive EW vs equal-risk (full-sample) vs causal
# ==============================================================================
def rolling_sharpe(pnl, win=252, annual=ANNUAL_D):
    r = pnl.rolling(win)
    return (r.mean() * annual) / (r.std(ddof=0) * np.sqrt(annual))

plt.figure(figsize=(12, 4))
rolling_sharpe(pnl_eqrisk_causal).plot(label="Equal-Risk (causal)", lw=2.4)
rolling_sharpe(pnl_eqrisk_fs).plot(label="Equal-Risk (inv-vol, full-sample)",
                                   lw=2.0, ls="--")
rolling_sharpe(pnl_eq_naive).plot(label="Naive EW (raw spread-$)",
                                  lw=1.4, ls=":", alpha=0.7)
plt.axhline(0, lw=1, ls="--", color="gray")
plt.title("Rolling 1Y Sharpe: OU Pair Portfolios")
plt.ylabel("Sharpe"); plt.legend(); plt.tight_layout(); plt.show()

In [ ]:
# ==============================================================================
# 3C. Return & risk contribution (equal-risk weights, full-sample)
# ==============================================================================
pairs = pnl_wide.columns
w = w_invvol.reindex(pairs).values            # equal-risk (inv-vol) weights
ann_pnl_vec = (pnl_wide.mean(axis=0) * ANNUAL_D).values
cov = pnl_wide.cov().values * ANNUAL_D

mu_port  = float(w @ ann_pnl_vec)
vol_port = float(np.sqrt(w @ cov @ w))
ret_contr  = (w * ann_pnl_vec / mu_port) if mu_port != 0 else np.zeros(len(w))
risk_contr = (w * (cov @ w) / vol_port / vol_port) if vol_port > 0 else np.zeros(len(w))

contrib = (pd.DataFrame({"Pair": pairs, "Weight": w,
                         "ReturnContribution": ret_contr,
                         "RiskContribution": risk_contr})
           .set_index("Pair"))

print("[INFO] Top 10 by RISK contribution (equal-risk weights):")
print(contrib.sort_values("RiskContribution", ascending=False).head(10).round(3).to_string())

In [ ]:
# ==============================================================================
# 3D. Turnover & trading intensity (from exported trades_table)
# ==============================================================================
years_total = len(pnl_wide) / ANNUAL_D
turn = (trades.groupby("pair").size().rename("TradesTotal").to_frame()
        .assign(TradesPerYear=lambda d: d["TradesTotal"] / years_total)
        .sort_values("TradesPerYear", ascending=False))

print("[INFO] Highest-turnover OU pairs (completed trades):")
print(turn.head(10).round(2).to_string())

plt.figure(figsize=(7, 4))
sns.histplot(turn["TradesPerYear"], bins=15, kde=True)
plt.title("Distribution of Trades per Year (OU Pair Strategies)")
plt.xlabel("Trades per Year"); plt.ylabel("Number of Pairs")
plt.tight_layout(); plt.show()

## Appendix Takeaways

**Naive EW is a units artifact, not a result.** Averaging raw spread-dollar
PnL gives a portfolio dominated by the few high-β pairs (see §1); its
headline Sharpe reflects those names, not a diversified book.

**Risk normalization restores breadth.** Scaling each pair to comparable
volatility — whether full-sample inverse-vol or the causal expanding-vol
variant — produces a portfolio whose return and risk are spread across the
set, and whose effective breadth (§3A.1) is the honest measure of
diversification. The causal version is the one to trust for any
forward-looking read, since its weights use only past information.

**Scope.** These are benchmarks for the *unconditioned* OU baseline. Real,
capital-scaled allocation belongs in NB04 on the AI-conditioned book; this
appendix exists to characterize the baseline and to make the PnL-scale
caveat explicit before that stage.